# FinRL #1463: fundamentals keyed on the period end, back-filled across tickers

https://github.com/AI4Finance-Foundation/FinRL/issues/1463

The report's minimal reproduction: ratios keyed on Compustat `datadate` (fiscal period end), then
`bfill` over the whole frame. Each quarter's EPS shows up before the quarter ends, and the last
ticker's tail is filled from the next ticker.

In [1]:
import datetime as dt, itertools
import pandas as pd
import fin_pit as pit

dates = pd.date_range("2020-01-01", "2020-07-10")
ratios = pd.DataFrame({"date": pd.to_datetime(["2020-03-31", "2020-06-30"] * 2),
                       "tic": ["AAA", "AAA", "BBB", "BBB"], "EPS": [1.0, 2.0, 10.0, 20.0]})
grid = pd.DataFrame(list(itertools.product(dates, ["AAA", "BBB"])), columns=["date", "tic"])
full = grid.merge(ratios, how="left", on=["date", "tic"]).sort_values(["tic", "date"]).bfill(axis="rows")
full[(full.tic == "AAA") & full.date.isin(pd.to_datetime(["2020-01-02", "2020-04-01", "2020-07-05"]))]

,date,tic,EPS
2,2020-01-02,AAA,1.0
182,2020-04-01,AAA,2.0
372,2020-07-05,AAA,10.0


## With fin-pit: join on availability, per ticker

Key each quarter on its report date (Compustat `rdq`; here synthetic dates about a month after the
period end), available at the end of that day in New York, and join each trading day's 16:00
decision to the newest report known by then.

In [2]:
ratios["rdq"] = pd.to_datetime(["2020-04-28", "2020-07-28", "2020-04-30", "2020-07-30"])   # synthetic
ratios["available_at"] = [pit.to_utc(dt.datetime.combine(d.date(), dt.time(23, 59, 59)), "America/New_York") for d in ratios["rdq"]]
grid["decision_time"] = [pit.to_utc(dt.datetime.combine(d.date(), dt.time(16)), "America/New_York") for d in grid["date"]]
pit_full = pit.asof_join(grid, ratios.drop(columns=["date"]), "decision_time", by="tic")
pit_full[(pit_full.tic == "AAA") & pit_full.date.isin(pd.to_datetime(["2020-01-02", "2020-04-01", "2020-04-29", "2020-07-05"]))][["date", "tic", "EPS"]]

,date,tic,EPS
2,2020-01-02,AAA,NaN
182,2020-04-01,AAA,NaN
238,2020-04-29,AAA,1.0
372,2020-07-05,AAA,1.0
